<a href="https://colab.research.google.com/github/Hyounseo/Opensource-FISH/blob/main/Url_%ED%83%90%EC%A7%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import re
import pandas as pd
import numpy as np

# [2단계-A] 편집 거리(Levenshtein Distance) 직접 구현
# [2단계-A]편집 거리(Levenshtein Distance) 직접 구현
def get_levenshtein_distance(s1, s2):
    if len(s1) < len(s2):
        return get_levenshtein_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)

    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    return previous_row[-1]


# [2단계-B]539개 정상 도메인 DB 파일 로드
# ⚠️ 파일명이 'correct_domain.csv'이고 'FISH_dataset' 폴더 안에 있는지 꼭 확인하세요!
csv_path = '/content/drive/MyDrive/FISH_dataset/correct_domain.csv'

print("--- 📂 정식 도메인 화이트리스트 데이터셋 로드 시작 ---")
if os.path.exists(csv_path):
    # 첫 줄 naver.com 누락 방지를 위한 header=None 처리 패치 적용
    df_domains = pd.read_csv(csv_path, header=None, names=['official_domain'])
    official_domains_db = df_domains['official_domain'].dropna().tolist()
    print(f"✅ 데이터셋 로드 완료! 총 {len(official_domains_db)}개의 정식 도메인이 준비되었습니다.")
else:
    print(f"❌ [에러] CSV 파일을 찾을 수 없습니다: {csv_path}")
    print("💡 경로가 다를 경우 아래 백업 데이터셋으로 자동 대체하여 연산합니다.")
    official_domains_db = ["naver.com", "daum.net", "google.com", "kakaobank.com", "toss.im", "coupang.com"]

--- 📂 정식 도메인 화이트리스트 데이터셋 로드 시작 ---
✅ 데이터셋 로드 완료! 총 539개의 정식 도메인이 준비되었습니다.


In [4]:
# =====================================================================
# [최종 완결] 원본 주석 및 로직 100% 보존 - 도메인 매칭 버그 완벽 수정 버전
# =====================================================================
!pip install -q easyocr scikit-learn gradio

import os, re, easyocr, pandas as pd
from google.colab import files
from urllib.parse import urlparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import gradio as gr

# ---------------- 구글 드라이브 마운트 추가 ----------------
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ---------------- CSV 로드 ----------------
correct_domain_path = '/content/drive/MyDrive/FISH_dataset/correct_domain.csv'
df_domains = pd.read_csv(correct_domain_path, header=None, names=['official_domain'])
official_domains_db = df_domains['official_domain'].dropna().str.strip().str.lower().tolist()
official_domains_db = list(set(official_domains_db))

extracted_path = '/content/drive/MyDrive/FISH_dataset/extracted_data.csv'
df = pd.read_csv(extracted_path)

# ---------------- 도메인 정제 함수 ----------------
def normalize_url_text(text):
    text = text.lower()
    text = re.sub(r'\|\|+', '', text)          # || 제거
    text = re.sub(r'\.{2,}', '.', text)        # 연속된 점 줄이기
    text = re.sub(r'[^a-z0-9:/.\-]', '', text) # 불필요한 특수문자 제거
    if text.startswith("http") and not text.startswith("http://"):
        text = text.replace("http", "http://", 1)
    text = re.sub(r'\bww+\.?', 'www.', text)   # www 보정
    return text

def extract_domain(text):
    text_norm = normalize_url_text(text)
    # urlparse가 netloc을 올바르게 추출할 수 있도록 앞의 스키마 보정 구조 적용
    if not text_norm.startswith("http://") and not text_norm.startswith("https://"):
        parsed = urlparse("http://" + text_norm)
    else:
        parsed = urlparse(text_norm)
    return parsed.netloc if parsed.netloc else text_norm

# ---------------- 학습 데이터 도메인 생성 ----------------
df['domain'] = df['text'].astype(str).apply(extract_domain)

vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2,4))
X = vectorizer.fit_transform(df['domain'].astype(str))
y = df['label']
model = LogisticRegression(max_iter=1000)
model.fit(X, y)

# ---------------- 판정 함수 ----------------
def classify_text(text):
    domain = extract_domain(text)

    # 0. 레이블 CSV 직접 매칭
    match = df.loc[df['domain'] == domain]
    if not match.empty:
        label = match['label'].values[0]
        if label == 0:
            return ("✅ 정상", domain, "CSV 직접 매칭", 0.0)
        else:
            return ("🚨 피싱 위험", domain, "CSV 직접 매칭", 100.0)

    # 1. 화이트리스트 비교
    if domain in official_domains_db:
        return ("✅ 정상", domain, "화이트리스트 등록", 0.0)

    # 2. CSV 학습 기반 판정
    X_test = vectorizer.transform([domain])
    pred = model.predict(X_test)[0]
    prob = model.predict_proba(X_test)[0][pred]
    if pred == 1:
        return ("🚨 피싱 위험", domain, "CSV 학습 기반 판정", round(prob*100,1))
    else:
        return ("⚠️ 미등록 도메인", domain, "CSV 학습 기반 판정", round(prob*100,1))

# ---------------- 반복 업로드 -> Gradio 인터페이스 연동 ----------------
reader = easyocr.Reader(['ko','en'], gpu=True)

# 그라디오용 안전 래핑 함수 (리턴값 매핑 보정)
def web_phishing_scan(image):
    if image is None:
        return "⚠️ 시연할 웹사이트/문자 캡처 이미지를 업로드해 주세요."

    ocr_result = reader.readtext(image, detail=0)
    extracted_text = " ".join(ocr_result)

    if not extracted_text.strip():
        return "🍏 안전: 이미지 내 추출된 텍스트 없음"

    # 원본 판정 함수 실행 후 결과를 4개 변수로 정확히 쪼개서 받음
    status, clean, detail, prob = classify_text(extracted_text)

    # 원본 print 양식 스타일로 최종 화면 출력
    return f"⚖️ {status}\n🧼 {clean}\n🎯 위험도 {prob}%\n📝 {detail}"

# 그라디오 UI 레이아웃 조립
demo = gr.Interface(
    fn=web_phishing_scan,
    inputs=gr.Image(type="filepath", label="📸 웹사이트 캡처 이미지 업로드"),
    outputs=gr.Textbox(label="🚨 실시간 분석 결과", lines=5),
    title="🛡️ 오픈소스프로그래밍 프로젝트 시연 모듈",
    description="스크린샷 이미지를 올리면 내부 EasyOCR 엔진이 주소를 스캔하여 정식 도메인 DB 및 CSV 기반 학습 모델을 거쳐 실시간 판정을 내립니다.",
    theme="soft"
)

# 외부 터널 링크 개방 및 실행
demo.launch(share=True, inline=True)

Mounted at /content/drive
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b59198ec2a6d0708fd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
